# EMG Channel Reduction — Clustering

Groups the 24 sEMG channels into 8 clusters to identify redundant electrodes.
Selects one representative channel per cluster, reducing features from 192 → 64 dims.

**Pipeline**:
1. Load preprocessed subject data from `emg_datahandler`
2. Extract hand-crafted features (sliding window, 8 features × 24 channels)
3. Compute 24×24 channel correlation matrix
4. Agglomerative clustering (k=8, correlation distance)
5. Select representative channel per cluster, build reduced feature matrix

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

# ── Path to shared public/ utilities ──────────────────────────────────────
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
PUBLIC    = os.path.join(REPO_ROOT, 'public')
if PUBLIC not in sys.path:
    sys.path.insert(0, PUBLIC)

from emg_loader import load_all_subjects, extract_features_from_subjects

# ── Config ─────────────────────────────────────────────────────────────────
DATA_DIR       = '/Volumes/KRIS/data/UG_per_subject'
WINDOW_SIZE    = 250
WINDOW_SHIFT   = 50
SAMPLING_RATE  = 5120.0
N_CH_CLUSTERS  = 8
N_FEATURES_PER_CH = 8  # MAV, RMS, WL, ZC, SSC, VAR, MNF, MDF
GESTURE_NAMES  = ['G1', 'G2', 'G3', 'G6', 'G7', 'G8', 'G9']

print('Imports OK.')

In [ ]:
subjects = load_all_subjects(DATA_DIR)
print(f'Loaded {len(subjects)} subject(s)')

In [ ]:
X, y = extract_features_from_subjects(
    subjects,
    window_size   = WINDOW_SIZE,
    window_shift  = WINDOW_SHIFT,
    sampling_rate = SAMPLING_RATE,
)

print(f'Class distribution: { {GESTURE_NAMES[i]: int((y==i).sum()) for i in range(7)} }')

---
## Channel Reduction via Clustering

The 192 features are ordered as `[8 feats ch0 | 8 feats ch1 | ... | 8 feats ch23]`.
Channels that measure the same underlying muscle will have correlated feature vectors.
We group them and keep one representative per group.

In [ ]:
# Reshape X (N, 192) -> X_ch (N, 24, 8)
X_ch = X.reshape(-1, 24, N_FEATURES_PER_CH)  # (N, 24, 8)
X_ch_mean = X_ch.mean(axis=0)                 # (24, 8) — mean profile per channel

feat_names = ['MAV', 'RMS', 'WL', 'ZC', 'SSC', 'VAR', 'MNF', 'MDF']
print('Per-channel mean feature profiles:')
print('  Ch   ' + '  '.join(f'{n:>6}' for n in feat_names))
for i, row in enumerate(X_ch_mean):
    print(f'  {i+1:>2}   ' + '  '.join(f'{v:>6.3f}' for v in row))

In [ ]:
# ── 24x24 Pearson correlation matrix ─────────────────────────────────────────
# Summarise each channel as one scalar per window (mean across its 8 features)
channel_signals = X_ch.mean(axis=2)       # (N, 24)
corr_matrix = np.corrcoef(channel_signals.T)  # (24, 24)
np.fill_diagonal(corr_matrix, 1.0)

mask = ~np.eye(24, dtype=bool)
print(f'Off-diagonal corr  min={corr_matrix[mask].min():.3f}  '
      f'max={corr_matrix[mask].max():.3f}  '
      f'mean={corr_matrix[mask].mean():.3f}')

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, vmin=-1, vmax=1, cmap='RdBu_r')
plt.colorbar(im, ax=ax, label='Pearson r')
ch_ticks = [f'Ch{i+1}' for i in range(24)]
ax.set_xticks(range(24)); ax.set_xticklabels(ch_ticks, rotation=90, fontsize=8)
ax.set_yticks(range(24)); ax.set_yticklabels(ch_ticks, fontsize=8)
ax.set_title('Channel-Channel Pearson Correlation', fontsize=13)
plt.tight_layout()
plt.savefig('channel_correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# ── Agglomerative clustering (correlation distance, average linkage) ──────────
dist_matrix = np.clip(1.0 - corr_matrix, 0, None)
np.fill_diagonal(dist_matrix, 0.0)

agg = AgglomerativeClustering(n_clusters=N_CH_CLUSTERS,
                              metric='precomputed', linkage='average')
ch_cluster_labels = agg.fit_predict(dist_matrix)

print('Cluster composition:')
for c in range(N_CH_CLUSTERS):
    members = np.where(ch_cluster_labels == c)[0] + 1
    print(f'  Cluster {c}: channels {list(members)}')

In [ ]:
# ── Representative = channel closest to cluster centroid ──────────────────────
representative_channels = []

print(f'  {"Cluster":>7}  {"Members":>22}  Representative')
print('  ' + '-'*52)
for c in range(N_CH_CLUSTERS):
    members  = np.where(ch_cluster_labels == c)[0]
    centroid = X_ch_mean[members].mean(axis=0)
    rep_ch   = members[np.argmin(np.linalg.norm(X_ch_mean[members] - centroid, axis=1))]
    representative_channels.append(rep_ch)
    print(f'  {c:>7}  {str(list(members+1)):>22}  Ch{rep_ch+1} (idx {rep_ch})')

representative_channels = sorted(representative_channels)
print(f'\nFinal (0-indexed): {representative_channels}')
print(f'Final (1-indexed): {[c+1 for c in representative_channels]}')

In [ ]:
# ── Build reduced feature matrix (N, 64) ─────────────────────────────────────
rep_feat_indices = []
for ch in representative_channels:
    rep_feat_indices.extend(range(ch * N_FEATURES_PER_CH, (ch + 1) * N_FEATURES_PER_CH))

X_reduced = X[:, rep_feat_indices]
print(f'Original : {X.shape}   (N, 192)')
print(f'Reduced  : {X_reduced.shape}    (N, 64)')

np.save('representative_channels.npy', np.array(representative_channels))
np.save('rep_feat_indices.npy', np.array(rep_feat_indices))
print('Saved representative_channels.npy and rep_feat_indices.npy')
print('To reduce any feature matrix: X_new[:, rep_feat_indices]')

In [ ]:
# ── Dendrogram ────────────────────────────────────────────────────────────────
condensed_dist = squareform(dist_matrix)
Z = linkage(condensed_dist, method='average')

fig, ax = plt.subplots(figsize=(13, 5))
dendrogram(
    Z,
    labels=[f'Ch{i+1}' for i in range(24)],
    color_threshold=Z[-(N_CH_CLUSTERS - 1), 2],
    ax=ax, leaf_rotation=0, leaf_font_size=10,
)
ax.set_title(f'Channel Dendrogram — average linkage  (k={N_CH_CLUSTERS})', fontsize=13)
ax.set_xlabel('Channel')
ax.set_ylabel('Correlation distance  (1 - r)')

# Mark selected representatives with a star
for tick in ax.get_xticklabels():
    ch_0idx = int(tick.get_text().replace('Ch', '')) - 1
    if ch_0idx in representative_channels:
        x = tick.get_position()[0]
        ylim = ax.get_ylim()
        ax.text(x, ylim[0] - (ylim[1]-ylim[0])*0.06, u'\u2605',
                ha='center', va='top', fontsize=12, color='red',
                transform=ax.transData)

plt.tight_layout()
plt.savefig('channel_dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()
print('Red star = selected representative channel')